# 🧠 深度学习：超越线性的边界

湖北理工学院《人工智能理论》课程资料

📝 **本节内容概述：**
1. 🌙 **数据准备**：生成经典的“月亮型”二分类数据集，挑战线性模型的极限。
2. 📉 **逻辑回归的局限**：演示为什么单层线性模型无法处理复杂的几何分布。
3. ⚡ **激活函数的作用**：让线性模型学会拐弯。

## 🌙 1. 数据准备

在真实的物理世界或复杂的业务场景中，数据往往不是“线性可分”的（即不能用一把直尺划开）。
今天我们使用 `scikit-learn` 生成一个**月亮型 (Moon)** 数据集。这两个类别像两个交织在一起的月牙，任何直线都无法完美地将它们分离。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns
from IPython.display import clear_output

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 👇 生成月亮型数据
# ========================================================
# n_samples: 样本数, noise: 噪点程度 (越大越乱)
X, y = make_moons(n_samples=500, noise=0.15, random_state=42)

# ========================================================
# 👇 划分训练集和测试集
# ========================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 可视化原始数据
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='#FF5733', label='类别 0', edgecolors='k', alpha=0.7)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='#33FF57', label='类别 1', edgecolors='k', alpha=0.7)
plt.title("月亮型二分类原始数据分布")
plt.legend()
plt.show()

# 4. 转换为 PyTorch 张量
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

## 📉 2. 逻辑回归：线性模型的“撞墙”时刻

逻辑回归本质上是在学习一个**超平面（在二维就是一条直线）**。对于月亮型这种弯曲的数据，逻辑回归能做到最好也就是在中间切一刀，必然会漏掉大量的弯曲部分。

In [ ]:
# 定义一个简单的逻辑回归模型
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(2, 1) # 输入 2 维坐标，输出 1 个概率得分
        
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model_lr = LogisticRegressionModel()
criterion = nn.BCELoss() # 二元交叉熵损失
optimizer = optim.Adam(model_lr.parameters(), lr=0.01)

# 训练逻辑回归模型
epochs = 1000
for epoch in range(epochs):
    outputs = model_lr(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 可视化决策边界辅助函数
def plot_decision_boundary(model, X, y, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
    
    model.eval()
    with torch.no_grad():
        grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
        Z = model(grid_tensor)
        Z = (Z > 0.5).float().numpy().reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap=plt.cm.coolwarm, alpha=0.7)
    ax.set_title(title)

plot_decision_boundary(model_lr, X, y, "[X] 逻辑回归：受限于线性边界")
print("⚠️ 观察发现：逻辑回归无法弯曲其边界，分类效果较差。")

## 🏗️ 3. 神经网络模型设计

为了打破线性限制，我们需要引入**隐藏层 (Hidden Layers)** 和 **激活函数 (Activation Functions)**。神经网络就像是在数据空间中“揉面团”，通过多层变换将原本扭曲的空间拉直。

为了比较激活函数的作用，我们设计两个总**隐藏神经元数量均为 12 个**的模型，区别在于有无激活函数：

In [ ]:
# 1. 纯线性神经网络 (无激活函数)
class PureLinearNN(nn.Module):
    def __init__(self):
        super(PureLinearNN, self).__init__()
        # 即使堆叠两层 Linear，没有非线性激活函数，结果依然是线性的
        self.model = nn.Sequential(
            nn.Linear(2, 12),
            nn.Linear(12, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x)

# 2. 非线性神经网络 (带 ReLU 激活函数)
class NonLinearNN(nn.Module):
    def __init__(self):
        super(NonLinearNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 12),
            nn.ReLU(),
            nn.Linear(12, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x)

print("✅ 对比模型已定义：PureLinearNN (无激活) vs NonLinearNN (有激活)")

## ✨ 4. 激活函数的“魔力”：有无对比

为了直观理解激活函数的作用，我们对比一个**两层线性网络**（无激活函数）和一个**两层非线性网络**（带 ReLU）。
你会发现，如果没有非线性激活函数，无论堆叠多少层，模型依然只能画出一条直线。

In [ ]:
# 🚀 激活函数的动态演变过程展示
import torch.nn as nn
import torch.optim as optim

m_linear, m_nonlinear = PureLinearNN(), NonLinearNN()
opt_l = optim.Adam(m_linear.parameters(), lr=0.01)
opt_nl = optim.Adam(m_nonlinear.parameters(), lr=0.01)

epochs = 3000
display_step = 200 # 每 200 步刷新一次图像

for epoch in range(epochs):
    # --- 训练逻辑 ---
    opt_l.zero_grad()
    loss_l = nn.BCELoss()(m_linear(X_train_t), y_train_t)
    loss_l.backward()
    opt_l.step()
    
    opt_nl.zero_grad()
    loss_nl = nn.BCELoss()(m_nonlinear(X_train_t), y_train_t)
    loss_nl.backward()
    opt_nl.step()

    # --- 动态可视化逻辑 ---
    if (epoch + 1) % display_step == 0 or epoch == 0:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # 绘制决策边界
        plot_decision_boundary(m_linear, X, y, f"[Epoch {epoch+1}] 激活函数：空间在弯曲", ax=axes[0])
        plot_decision_boundary(m_nonlinear, X, y, f"[Epoch {epoch+1}] 无激活函数：依然是线性空间", ax=axes[1])
        
        plt.tight_layout()
        plt.show()
        plt.close() # 显式关闭画布释放内存
        
print("💡 结论：没有激活函数，深层网络退化为单层线性模型；有了激活函数，网络才具备了‘扭曲’空间的能力。")